# Advanced Problems with Solutions: Shared References and Mutability

This notebook builds advanced intuition for **shared references**, **object identity**, **mutability vs immutability**, **rebinding**, and **copying** in Python.

## Best practices emphasized
- Distinguish **same value** from **same object**.
- Use `is` for identity checks and `==` for value equality.
- Avoid assuming implementation-specific memory behaviors.
- Be explicit about when a function **mutates** an object versus **rebinds** a name.
- Use copies intentionally when shared state would be risky.


## Problem 1 — Identity vs Equality

Two objects can compare equal without being the same object.

For each pair below, predict:
1. The result of `==`
2. The result of `is`
3. Which cases are safe to reason about purely from Python semantics, and which may depend on implementation details


In [1]:
a = [1, 2, 3]
b = [1, 2, 3]
c = a

x = 'hello world'
y = 'hello world'

m = 256
n = 256

print('a == b:', a == b)
print('a is b:', a is b)
print('a == c:', a == c)
print('a is c:', a is c)
print('x == y:', x == y)
print('x is y:', x is y)
print('m == n:', m == n)
print('m is n:', m is n)

a == b: True
a is b: False
a == c: True
a is c: True
x == y: True
x is y: False
m == n: True
m is n: True


### Solution 1

- `a == b` is `True` because the lists have equal contents.
- `a is b` is `False` because they are distinct list objects.
- `a == c` and `a is c` are both `True` because `c` is just another reference to `a`.

For strings and small integers, `is` may sometimes be `True` due to interpreter optimizations such as interning or caching. However, **you should not rely on that for program logic**.

Best practice:
- use `==` to compare values
- use `is` only for identity-sensitive checks, such as `x is None`


## Problem 2 — Rebinding a Name vs Mutating an Object

Predict the final values and identities of the variables below.

Explain why the first operation does not affect `left`, while the second does affect `right`.

In [2]:
left = 'start'
alias_left = left

right = [1, 2, 3]
alias_right = right

alias_left = alias_left + '-end'
alias_right.append(4)

print('left       =', left)
print('alias_left =', alias_left)
print('right      =', right)
print('alias_right=', alias_right)

print(hex(id(left)))
print(hex(id(alias_left)))
print(hex(id(right)))
print(hex(id(alias_right)))

left       = start
alias_left = start-end
right      = [1, 2, 3, 4]
alias_right= [1, 2, 3, 4]
0x7ffca4a04f30
0x194d3ce5cb0
0x194d3cbba80
0x194d3cbba80


### Solution 2

Strings are immutable, so `alias_left = alias_left + '-end'` creates a **new string object** and rebinds only `alias_left`.

Lists are mutable, so `alias_right.append(4)` changes the **existing shared list object** in place. Since `right` and `alias_right` refer to the same list, both reflect the change.

Core idea:
- rebinding changes which object a name refers to
- mutation changes the internal state of an existing object


## Problem 3 — Shared References Inside Nested Containers

Nested structures make aliasing harder to spot.

Predict the final value of `matrix` and explain why changing one row changes another.

In [3]:
row = [0, 0]
matrix = [row, row, row]

matrix[0][1] = 99

print('matrix =', matrix)
print('same row object?', matrix[0] is matrix[1], matrix[1] is matrix[2])
print('row ids:', [hex(id(r)) for r in matrix])

matrix = [[0, 99], [0, 99], [0, 99]]
same row object? True True
row ids: ['0x194d3cba700', '0x194d3cba700', '0x194d3cba700']


### Solution 3

All three elements of `matrix` refer to the **same inner list object**.

So changing `matrix[0][1]` changes that one shared list, and the result is:

`[[0, 99], [0, 99], [0, 99]]`

This is a classic aliasing pitfall. A safer construction is to create distinct row objects, for example with a comprehension.


In [4]:
safe_matrix = [[0, 0] for _ in range(3)]
safe_matrix[0][1] = 99
print(safe_matrix)
print([hex(id(r)) for r in safe_matrix])

[[0, 99], [0, 0], [0, 0]]
['0x194d3ca9a00', '0x194d3cb8ec0', '0x194d3cbac40']


## Problem 4 — Function Calls and Shared State

Analyze what the caller sees after the function call.

For each line in `adjust`, state whether it mutates the original object or only changes a local reference.

In [5]:
def adjust(data):
    print('start id(data):', hex(id(data)))
    data['scores'].append(50)
    data['name'] = data['name'].upper()
    data = {'name': 'NEW', 'scores': [0]}
    print('end id(data):  ', hex(id(data)))
    return data

record = {'name': 'alice', 'scores': [10, 20]}
result = adjust(record)

print('record =', record)
print('result =', result)
print('same outer dict?', record is result)

start id(data): 0x194d3ce6b40
end id(data):   0x194d3ce7c80
record = {'name': 'ALICE', 'scores': [10, 20, 50]}
result = {'name': 'NEW', 'scores': [0]}
same outer dict? False


### Solution 4

- `data['scores'].append(50)` mutates the shared list inside the original dictionary.
- `data['name'] = data['name'].upper()` updates an entry in the original dictionary. The new uppercase string is a new object, but the dictionary itself is still the caller's dictionary.
- `data = {'name': 'NEW', 'scores': [0]}` rebinds the **local** name `data` to a new dictionary, which does not affect the caller's variable.

So:
- `record` becomes `{'name': 'ALICE', 'scores': [10, 20, 50]}`
- `result` is `{'name': 'NEW', 'scores': [0]}`
- `record is result` is `False`


## Problem 5 — Why Python Avoids Sharing Mutable Literals Automatically

Suppose a Python implementation automatically reused the same list object for identical list literals. Show why that would be dangerous.

Predict the output of the code below under normal Python semantics, then explain what would go wrong under such an imaginary optimization.

In [6]:
list_a = [1, 2, 3]
list_b = [1, 2, 3]

list_a.append(4)

print('list_a =', list_a)
print('list_b =', list_b)
print('same object?', list_a is list_b)

list_a = [1, 2, 3, 4]
list_b = [1, 2, 3]
same object? False


### Solution 5

Under normal Python semantics:
- `list_a` becomes `[1, 2, 3, 4]`
- `list_b` remains `[1, 2, 3]`
- `list_a is list_b` is `False`

If Python reused the same mutable list object for both literals, mutating `list_a` would unexpectedly mutate `list_b` too. That would make ordinary code unsafe and unpredictable.

This is why sharing immutable objects can be safe, while automatic sharing of mutable objects would usually be a bad idea.

## Problem 6 — Shallow Copies Preserve Some Sharing

A developer tries to avoid aliasing by copying a dictionary, but nested structures still behave unexpectedly.

Predict the outputs and explain the bug.

In [7]:
original = {
    'user': 'sam',
    'prefs': {
        'theme': 'dark',
        'alerts': ['email']
    }
}

clone = original.copy()
clone['prefs']['alerts'].append('sms')

print('original =', original)
print('clone    =', clone)
print('same prefs object?', original['prefs'] is clone['prefs'])
print('same alerts object?', original['prefs']['alerts'] is clone['prefs']['alerts'])

original = {'user': 'sam', 'prefs': {'theme': 'dark', 'alerts': ['email', 'sms']}}
clone    = {'user': 'sam', 'prefs': {'theme': 'dark', 'alerts': ['email', 'sms']}}
same prefs object? True
same alerts object? True


### Solution 6

`dict.copy()` makes only a **shallow copy** of the outer dictionary.

So `original` and `clone` are different dictionaries, but they still share the same nested `prefs` dictionary and the same nested `alerts` list.

As a result, appending `'sms'` through `clone` also changes `original`.

When nested mutable objects must be independent, use `copy.deepcopy()` or reconstruct the data structure explicitly.

In [8]:
from copy import deepcopy

original = {
    'user': 'sam',
    'prefs': {
        'theme': 'dark',
        'alerts': ['email']
    }
}

safe_clone = deepcopy(original)
safe_clone['prefs']['alerts'].append('sms')

print('original   =', original)
print('safe_clone =', safe_clone)
print('same prefs object?', original['prefs'] is safe_clone['prefs'])

original   = {'user': 'sam', 'prefs': {'theme': 'dark', 'alerts': ['email']}}
safe_clone = {'user': 'sam', 'prefs': {'theme': 'dark', 'alerts': ['email', 'sms']}}
same prefs object? False


## Problem 7 — Aliasing Through Repetition Syntax

The following pattern is common and subtle:

```python
grid = [[0] * 3] * 4
```

Predict the result of the code below and explain why it happens.

In [9]:
grid = [[0] * 3] * 4
grid[2][1] = 7

print(grid)
print([hex(id(row)) for row in grid])

[[0, 7, 0], [0, 7, 0], [0, 7, 0], [0, 7, 0]]
['0x194d3cb9180', '0x194d3cb9180', '0x194d3cb9180', '0x194d3cb9180']


### Solution 7

List repetition here duplicates **references**, not independent inner lists.

All four rows refer to the same row object, so changing one row changes them all. The result is:

`[[0, 7, 0], [0, 7, 0], [0, 7, 0], [0, 7, 0]]`

Safer version:

```python
grid = [[0] * 3 for _ in range(4)]
```


In [10]:
safe_grid = [[0] * 3 for _ in range(4)]
safe_grid[2][1] = 7
print(safe_grid)
print([hex(id(row)) for row in safe_grid])

[[0, 0, 0], [0, 0, 0], [0, 7, 0], [0, 0, 0]]
['0x194d3cb9fc0', '0x194d3cb9d00', '0x194d3cdbfc0', '0x194d3cb9f00']


## Problem 8 — Designing Around Shared Mutable State

A logging helper should keep track of messages, but a bug causes logs from unrelated calls to mix together.

1. Explain the bug.
2. Fix it.
3. State when shared state would be intentional rather than a problem.


In [11]:
def log_message(msg, bucket=[]):
    bucket.append(msg)
    return bucket

print(log_message('start'))
print(log_message('load-config'))
print(log_message('fresh-run', ['isolated']))
print(log_message('shutdown'))

['start']
['start', 'load-config']
['isolated', 'fresh-run']
['start', 'load-config', 'shutdown']


### Solution 8

The default list `bucket=[]` is created once at function definition time, then shared across later calls that omit the argument.

That produces accidental shared mutable state.

A safe fix is:

In [12]:
def log_message_fixed(msg, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(msg)
    return bucket

print(log_message_fixed('start'))
print(log_message_fixed('load-config'))
print(log_message_fixed('fresh-run', ['isolated']))
print(log_message_fixed('shutdown'))

['start']
['load-config']
['isolated', 'fresh-run']
['shutdown']


Shared state would be intentional if you were deliberately implementing a cache, memoization table, registry, or persistent accumulator. In those cases, the sharing should be explicit and documented.

## Problem 9 — Trace the Aliases

Without running the code first, determine which names refer to the same object after each step.

Then verify your reasoning by checking identities.

In [13]:
a = [1]
b = a
c = a.copy()
d = [a, c]

a.append(2)
c.append(3)
b = b + [4]

print('a =', a)
print('b =', b)
print('c =', c)
print('d =', d)

print('a is d[0]:', a is d[0])
print('c is d[1]:', c is d[1])
print('a is b:', a is b)
print('a == b:', a == b)

a = [1, 2]
b = [1, 2, 4]
c = [1, 3]
d = [[1, 2], [1, 3]]
a is d[0]: True
c is d[1]: True
a is b: False
a == b: False


### Solution 9

Initially:
- `b` aliases `a`
- `c` is a distinct shallow copy of `a`
- `d[0]` refers to `a`, and `d[1]` refers to `c`

Then:
- `a.append(2)` mutates the object shared by `a`, `b`, and `d[0]`
- `c.append(3)` mutates only `c` and therefore `d[1]`
- `b = b + [4]` creates a new list and rebinds `b`, breaking its alias with `a`

Final values:
- `a == [1, 2]`
- `b == [1, 2, 4]`
- `c == [1, 3]`
- `d == [[1, 2], [1, 3]]`


## Problem 10 — API Review: Should This Function Mutate?

Consider this function for preparing a report payload:

```python
def prepare(payload):
    payload['tags'].append('processed')
    payload['status'] = 'done'
    return payload
```

Answer the following:
1. What risks does this design create?
2. When might mutation be acceptable?
3. Write a safer non-mutating version.


In [14]:
def prepare_safe(payload):
    return {
        **payload,
        'tags': [*payload['tags'], 'processed'],
        'status': 'done'
    }

original = {'tags': ['draft'], 'status': 'new'}
prepared = prepare_safe(original)

print('original =', original)
print('prepared =', prepared)
print('same dict?', original is prepared)
print('same tags list?', original['tags'] is prepared['tags'])

original = {'tags': ['draft'], 'status': 'new'}
prepared = {'tags': ['draft', 'processed'], 'status': 'done'}
same dict? False
same tags list? False


### Solution 10

The mutating design can create surprising side effects when callers expect their input data to remain unchanged, especially if the same payload is shared across multiple parts of a program.

Mutation may be acceptable when:
- performance is critical
- the function contract clearly documents the side effect
- callers explicitly opt into in-place updates

The safer version returns a new dictionary and a new `tags` list, avoiding shared-state surprises.

## Summary

Key rule:

> Names are references to objects. Bugs around mutability usually come from forgetting which names share the same object.

When analyzing code, ask:
1. Are these two names bound to the same object?
2. Is this operation mutating an object or rebinding a name?
3. Is there nested shared mutable state?
4. Would a copy make the behavior safer or clearer?
